In [ ]:
# Install packages (CPU time, keep it short)
# The L4 is sm_89 / CUDA 12 capable, so we use the latest prebuilt torch wheels.
!pip install -q --upgrade torch torchvision
!pip install -q segmentation-models-pytorch==0.3.3 albumentations==2.0.8 timm
!pip install -q pycocotools scikit-image scipy opencv-python-headless pandas tqdm kaggle kagglehub

In [ ]:
# Download the competition data
# Get your token from C:\Users\srik2\.kaggle\access_token (the KGAT_... string)
# Do NOT save this notebook with the token in any cell.
import os, getpass
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Paste your Kaggle API token (from .kaggle/access_token): ')
import kagglehub
data_root = kagglehub.competition_download('filament-segmentation-2026', output_dir='./data', force_download=True)
print('Downloaded to:', data_root)

In [ ]:
# Locate the actual base path and set the config env vars
import os
for p in [data_root, os.path.join(data_root, 'MAGFiLO_1.0_Kaggle_2026')]:
    if p and os.path.isdir(os.path.join(p, 'train', 'train_images')):
        os.environ['FILAMENT_BASE_PATH'] = p
        break
os.environ['FILAMENT_PRESET'] = 'balanced'
print('FILAMENT_BASE_PATH =', os.environ['FILAMENT_BASE_PATH'])

In [ ]:
# L4-optimized 5-fold run with time budget guard
import os, time
import filament_top50_single as ft

start_t = time.time()
BUDGET_H = 4.8   # stop 12 minutes before the 5h free quota
probs_dirs = []

for fold in range(5):
    elapsed_h = (time.time() - start_t) / 3600
    if elapsed_h >= BUDGET_H:
        print(f'Budget guard: {elapsed_h:.2f}h elapsed, stopping before 5h.')
        break

    print(f'
{"="*60}
FOLD {fold}/4 (elapsed {elapsed_h:.2f}h)
{"="*60}')
    cfg = ft.apply_preset(ft.CFG, 'balanced')

    # Use the full L4: larger batch + more workers
    cfg.batch_size = 4       # 24 GB L4 can hold this with grad_checkpoint on
    cfg.accum = 2            # effective batch = 8 (same as balanced default)
    cfg.num_workers = 4
    # Drop multi-scale TTA so 5 folds fit in the 5h budget.
    # D4 (8 flips) is kept; that is still a strong TTA.
    cfg.tta_scales = (1.0,)
    cfg.fold = fold

    res = ft.run(cfg, do_train=True, do_search=True, do_submit=True)
    probs_dirs.append(os.path.join(cfg.prob_dir, f'test_fold_{fold}'))

print('
Completed', len(probs_dirs), 'folds. Prob map dirs:')
print(probs_dirs)

In [ ]:
# Ensemble the saved probability maps into one submission
import os
import filament_top50_single as ft

ens_dir = os.path.join('filament_out', 'probs', 'ensemble')
ft.ensemble_probs(probs_dirs, ens_dir)

# Use the threshold/min-area from the last completed fold (best saved post-processing)
thr = res['threshold'] if 'res' in globals() else ft.CFG.threshold
ma  = res['min_area']  if 'res' in globals() else ft.CFG.min_area
final_csv = f'filament_out/submissions/submission_ensemble_thr{thr:.2f}_ma{ma}.csv'
final = ft.submission_from_prob_dir(ft.CFG, ens_dir, threshold=thr, min_area=ma, out_csv=final_csv)
print('Final ensemble submission:', final_csv)